In [ ]:
!pip install transformers torch youtube-transcript-api scikit-learn nltk gensim sentence-transformers rouge-score
!pip install spacy
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Hierarchical Summarization of Video Transcripts
Combines Topic Modeling, Extractive, and Abstractive Summarization Techniques


In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
import spacy

# Topic Modeling
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from gensim import corpora
from gensim.models import LdaModel

# Transformers for Summarization
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    T5ForConditionalGeneration,
    T5Tokenizer,
    pipeline
)

# Sentence Embeddings
from sentence_transformers import SentenceTransformer, util

# Evaluation
from rouge_score import rouge_scorer

# YouTube Transcript
from youtube_transcript_api import YouTubeTranscriptApi

import torch
import re

# Download required NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except:
    pass

class VideoTranscriptSummarizer:
    """
    Hierarchical summarization system for video transcripts
    """

    def __init__(self, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        print(f"Using device: {self.device}")

        # Load models
        print("Loading models...")
        self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

        # For abstractive summarization
        self.bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
        self.bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(self.device)

        # Load spacy for sentence segmentation
        self.nlp = spacy.load('en_core_web_sm')
        self.stop_words = set(stopwords.words('english'))

        print("Models loaded successfully!")

    def fetch_youtube_transcript(self, video_id):
        """
        Fetch transcript from YouTube video
        """
        try:
            transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
            transcript_text = ' '.join([entry['text'] for entry in transcript_list])
            timestamps = [(entry['start'], entry['text']) for entry in transcript_list]
            return transcript_text, timestamps
        except Exception as e:
            print(f"Error fetching transcript: {e}")
            return None, None

    def preprocess_text(self, text):
        """
        Clean and preprocess transcript text
        """
        # Remove special characters and extra whitespace
        text = re.sub(r'\[.*?\]', '', text)  # Remove [Music], [Applause], etc.
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        return text

    def segment_into_chunks(self, text, chunk_size=300):
        """
        Segment transcript into semantic chunks (by sentences)
        """
        sentences = sent_tokenize(text)

        # If we have very few sentences, make smaller chunks
        if len(sentences) < 10:
            chunk_size = 100

        chunks = []
        current_chunk = []
        current_length = 0

        for sentence in sentences:
            sentence_length = len(sentence.split())

            if current_length + sentence_length > chunk_size and current_chunk:
                chunks.append(' '.join(current_chunk))
                current_chunk = [sentence]
                current_length = sentence_length
            else:
                current_chunk.append(sentence)
                current_length += sentence_length

        if current_chunk:
            chunks.append(' '.join(current_chunk))

        # Ensure we have at least 3 chunks for topic modeling
        if len(chunks) < 3:
            # Split the text into roughly equal parts
            words = text.split()
            words_per_chunk = max(50, len(words) // 3)
            chunks = []
            for i in range(0, len(words), words_per_chunk):
                chunk = ' '.join(words[i:i + words_per_chunk])
                if chunk.strip():
                    chunks.append(chunk)

        return chunks

    def topic_modeling_lda(self, chunks, n_topics=5):
        """
        Perform LDA topic modeling on chunks
        """
        n_chunks = len(chunks)

        # For very small corpora, adjust parameters to avoid errors
        if n_chunks < 5:
            min_df = 1
            max_df = 1.0  # Include all documents
        elif n_chunks < 20:
            min_df = 1
            max_df = 0.99
        else:
            min_df = 2
            max_df = 0.85

        # Create document-term matrix
        try:
            vectorizer = CountVectorizer(
                max_features=min(1000, n_chunks * 50),
                stop_words='english',
                min_df=min_df,
                max_df=max_df,
                lowercase=True,
                token_pattern=r'\b[a-zA-Z]{3,}\b'  # Words with at least 3 letters
            )
            doc_term_matrix = vectorizer.fit_transform(chunks)
        except ValueError as e:
            # Fallback to most lenient settings
            print(f"Adjusting vectorizer parameters due to: {e}")
            vectorizer = CountVectorizer(
                max_features=500,
                stop_words='english',
                min_df=1,
                max_df=1.0
            )
            doc_term_matrix = vectorizer.fit_transform(chunks)

        # Adjust n_topics based on number of chunks (can't have more topics than documents)
        actual_n_topics = min(n_topics, max(2, n_chunks - 1))

        # LDA model
        lda = LatentDirichletAllocation(
            n_components=actual_n_topics,
            random_state=42,
            max_iter=20,
            learning_method='batch'
        )
        lda.fit(doc_term_matrix)

        # Get topics
        feature_names = vectorizer.get_feature_names_out()
        topics = []
        for topic_idx, topic in enumerate(lda.components_):
            top_words_idx = topic.argsort()[-10:][::-1]
            top_words = [feature_names[i] for i in top_words_idx]
            topics.append({
                'topic_id': topic_idx,
                'top_words': top_words,
                'weights': topic[top_words_idx]
            })

        # Assign chunks to topics
        chunk_topics = lda.transform(doc_term_matrix)
        chunk_assignments = chunk_topics.argmax(axis=1)

        return topics, chunk_assignments, chunk_topics

    def extractive_summarization(self, chunks, chunk_topics, top_k=3):
        """
        Extractive summarization using sentence embeddings and clustering
        """
        # Get embeddings for all chunks
        embeddings = self.sentence_model.encode(chunks, convert_to_tensor=True)

        # Group chunks by topic
        topic_chunks = defaultdict(list)
        for idx, topic_id in enumerate(chunk_topics):
            topic_chunks[topic_id].append((idx, chunks[idx], embeddings[idx]))

        # Extract representative sentences from each topic
        extractive_summary = []

        for topic_id, topic_data in topic_chunks.items():
            if not topic_data:
                continue

            # Get embeddings for this topic
            topic_embeddings = torch.stack([item[2] for item in topic_data])

            # Find most central chunk (closest to centroid)
            centroid = torch.mean(topic_embeddings, dim=0)
            similarities = util.cos_sim(centroid.unsqueeze(0), topic_embeddings)[0]

            # Get top-k most representative chunks
            top_indices = similarities.argsort(descending=True)[:top_k]

            for idx in top_indices:
                original_idx = topic_data[idx.item()][0]
                extractive_summary.append({
                    'topic_id': topic_id,
                    'chunk_idx': original_idx,
                    'text': topic_data[idx.item()][1],
                    'score': similarities[idx].item()
                })

        # Sort by original order
        extractive_summary.sort(key=lambda x: x['chunk_idx'])

        return extractive_summary

    def abstractive_summarization(self, text, max_length=150, min_length=50):
        """
        Generate abstractive summary using BART
        """
        inputs = self.bart_tokenizer(text, max_length=1024, truncation=True,
                                     return_tensors='pt').to(self.device)

        summary_ids = self.bart_model.generate(
            inputs['input_ids'],
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

        summary = self.bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return summary

    def hierarchical_summarization(self, text, n_topics=5, extractive_per_topic=2):
        """
        Complete hierarchical summarization pipeline
        """
        print("\n=== Starting Hierarchical Summarization ===\n")

        # Step 1: Preprocess
        print("Step 1: Preprocessing text...")
        clean_text = self.preprocess_text(text)

        # Step 2: Segment into chunks
        print("Step 2: Segmenting into chunks...")
        chunks = self.segment_into_chunks(clean_text, chunk_size=300)
        print(f"Created {len(chunks)} chunks")

        # Adjust n_topics based on number of chunks
        n_topics = min(n_topics, max(2, len(chunks) - 1))

        # Step 3: Topic modeling
        print(f"Step 3: Performing topic modeling ({n_topics} topics)...")
        topics, chunk_assignments, chunk_topics = self.topic_modeling_lda(
            chunks, n_topics=n_topics
        )

        print("\nDiscovered Topics:")
        for topic in topics:
            print(f"  Topic {topic['topic_id']}: {', '.join(topic['top_words'][:5])}")

        # Step 4: Extractive summarization
        print(f"\nStep 4: Extractive summarization (top {extractive_per_topic} per topic)...")
        extractive_summary = self.extractive_summarization(
            chunks, chunk_assignments, top_k=extractive_per_topic
        )

        # Combine extractive summaries
        extractive_text = ' '.join([item['text'] for item in extractive_summary])

        # If extractive text is too short, add more context
        if len(extractive_text.split()) < 100:
            extractive_text = ' '.join(chunks[:min(3, len(chunks))])

        # Step 5: Abstractive summarization
        print("Step 5: Generating abstractive summary...")
        abstractive_summary = self.abstractive_summarization(extractive_text)

        results = {
            'topics': topics,
            'n_chunks': len(chunks),
            'extractive_summary': extractive_summary,
            'extractive_text': extractive_text,
            'abstractive_summary': abstractive_summary,
            'chunk_assignments': chunk_assignments.tolist()
        }

        print("\n=== Summarization Complete ===\n")
        return results

    def evaluate_summary(self, generated_summary, reference_summary):
        """
        Evaluate summary using ROUGE scores
        """
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'],
                                          use_stemmer=True)
        scores = scorer.score(reference_summary, generated_summary)

        return {
            'rouge1': scores['rouge1'].fmeasure,
            'rouge2': scores['rouge2'].fmeasure,
            'rougeL': scores['rougeL'].fmeasure
        }

    def display_results(self, results):
        """
        Display summarization results
        """
        print("\n" + "="*80)
        print("HIERARCHICAL SUMMARIZATION RESULTS")
        print("="*80)

        print(f"\n📊 STATISTICS:")
        print(f"  • Total chunks: {results['n_chunks']}")
        print(f"  • Topics identified: {len(results['topics'])}")
        print(f"  • Extractive sentences: {len(results['extractive_summary'])}")

        print(f"\n🎯 TOPICS DISCOVERED:")
        for topic in results['topics']:
            print(f"\n  Topic {topic['topic_id']}:")
            print(f"    Keywords: {', '.join(topic['top_words'][:8])}")

        print(f"\n📝 EXTRACTIVE SUMMARY (Key Segments):")
        print("-" * 80)
        for i, item in enumerate(results['extractive_summary'][:5], 1):
            print(f"\n  [{i}] Topic {item['topic_id']} (Score: {item['score']:.3f})")
            print(f"      {item['text'][:300]}...")

        print(f"\n✨ FINAL ABSTRACTIVE SUMMARY:")
        print("-" * 80)
        print(f"\n{results['abstractive_summary']}")
        print("\n" + "="*80)


In [ ]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """
    Main execution function
    """
    # Initialize summarizer
    summarizer = VideoTranscriptSummarizer()

    # Example 1: Sample transcript (replace with your own)
    sample_transcript = """
    Welcome to this comprehensive lecture on machine learning and artificial intelligence.
    Today we'll discuss supervised learning algorithms and their applications in real-world scenarios.
    Supervised learning is a fundamental type of machine learning where we train models on labeled data.
    The training process involves showing the algorithm examples with correct answers so it can learn patterns.

    The most common supervised learning algorithms include linear regression, logistic regression,
    decision trees, random forests, support vector machines, and neural networks. Each has unique strengths.
    Linear regression is primarily used for predicting continuous numerical values like house prices or temperatures.
    It works by finding the best fitting line through the data points using least squares optimization.
    The algorithm minimizes the sum of squared errors between predicted and actual values.

    Logistic regression, despite its name, is actually used for classification problems rather than regression.
    It predicts the probability of an instance belonging to a particular class using the sigmoid function.
    Common applications include spam detection, disease diagnosis, and customer churn prediction.
    The model outputs probabilities between 0 and 1, making it interpretable for binary classification.

    Decision trees are versatile algorithms that can handle both classification and regression tasks effectively.
    They work by recursively splitting the data based on feature values to create a tree-like structure.
    Each internal node represents a decision based on a feature, and leaf nodes represent predictions.
    Decision trees are easy to interpret and visualize, making them popular in business applications.
    Random forests extend this concept by creating multiple decision trees and aggregating their predictions.

    Support vector machines are powerful algorithms for classification that find optimal decision boundaries.
    They work by finding the hyperplane that maximizes the margin between different classes.
    SVMs can handle non-linear relationships using kernel tricks like RBF, polynomial, and sigmoid kernels.
    They are particularly effective in high-dimensional spaces and when classes are well-separated.

    Neural networks represent one of the most powerful approaches in modern machine learning.
    They are inspired by biological neurons in the human brain and consist of layers of interconnected nodes.
    Each connection has a weight that is adjusted during training through backpropagation algorithm.
    Neural networks excel at learning complex, non-linear patterns in large datasets.

    Deep learning is a subset of machine learning that uses neural networks with many hidden layers.
    These deep architectures can learn hierarchical representations of data automatically.
    Convolutional neural networks are specialized for image processing and computer vision tasks.
    Recurrent neural networks excel at sequential data like text, speech, and time series.

    Training these models requires large amounts of labeled data and significant computational resources.
    The training process involves iteratively adjusting model parameters to minimize a loss function.
    Techniques like gradient descent and its variants are used for optimization.
    Regularization methods like L1 and L2 help prevent overfitting on training data.

    In recent years, transfer learning has become increasingly popular in the machine learning community.
    This approach allows us to use pre-trained models as starting points for new, related tasks.
    Instead of training from scratch, we can fine-tune existing models on our specific dataset.
    This dramatically reduces training time and data requirements while often improving performance.

    Model evaluation is crucial for understanding how well our algorithms perform on unseen data.
    Common metrics include accuracy, precision, recall, F1-score for classification tasks.
    For regression problems, we use metrics like mean squared error, mean absolute error, and R-squared.
    Cross-validation techniques help ensure our models generalize well to new data.

    Feature engineering remains an important aspect of building effective machine learning models.
    This involves selecting, transforming, and creating relevant features from raw data.
    Domain knowledge often plays a crucial role in identifying useful features.
    Automated feature selection techniques can help identify the most informative variables.

    Hyperparameter tuning is the process of finding optimal configuration for model parameters.
    Grid search and random search are common approaches for exploring the parameter space.
    More advanced methods like Bayesian optimization can find good hyperparameters more efficiently.
    Proper hyperparameter selection can significantly improve model performance.

    This concludes our comprehensive overview of supervised learning algorithms and their applications.
    We've covered the major algorithm families, their strengths and weaknesses, and practical considerations.
    Understanding these fundamentals is essential for anyone working in data science and machine learning.
    The field continues to evolve rapidly with new techniques and applications emerging constantly.
    """

    print("Example 1: Using sample transcript")
    print("="*80)
    results = summarizer.hierarchical_summarization(
        sample_transcript,
        n_topics=3,
        extractive_per_topic=2
    )
    summarizer.display_results(results)

    # Example 2: YouTube video (uncomment to use)
    """
    print("\n\nExample 2: YouTube video transcript")
    print("="*80)
    video_id = "dQw4w9WgXcQ"  # Replace with actual video ID
    transcript_text, timestamps = summarizer.fetch_youtube_transcript(video_id)

    if transcript_text:
        results = summarizer.hierarchical_summarization(
            transcript_text,
            n_topics=5,
            extractive_per_topic=2
        )
        summarizer.display_results(results)
    """

    return summarizer, results


if __name__ == "__main__":
    summarizer, results = main()


Using device: cuda
Loading models...
Models loaded successfully!
Example 1: Using sample transcript

=== Starting Hierarchical Summarization ===

Step 1: Preprocessing text...
Step 2: Segmenting into chunks...
Created 3 chunks
Step 3: Performing topic modeling (2 topics)...

Discovered Topics:
  Topic 0: techniques, model, applications, search, selection
  Topic 1: learning, data, like, regression, training

Step 4: Extractive summarization (top 2 per topic)...
Step 5: Generating abstractive summary...

=== Summarization Complete ===


HIERARCHICAL SUMMARIZATION RESULTS

📊 STATISTICS:
  • Total chunks: 3
  • Topics identified: 2
  • Extractive sentences: 3

🎯 TOPICS DISCOVERED:

  Topic 0:
    Keywords: techniques, model, applications, search, selection, hyperparameter, learning, methods

  Topic 1:
    Keywords: learning, data, like, regression, training, machine, decision, networks

📝 EXTRACTIVE SUMMARY (Key Segments):
-----------------------------------------------------------------